In [136]:
import pandas as pd
import random
from datetime import timedelta


# Read data from a JSON file
file_path = 'stoparrivals.json'  # Replace with your actual JSON file path
df = pd.read_json(file_path)

print(df)

# Rename column
df.rename(columns={"Time": "ExpectedTime"}, inplace=True)

# Display updated DataFrame
print(df)

       StopID                             StopName        Lat        Lon  \
0        5296                           TC- Gate 6  43.158369 -77.608353   
1        5308                          TC- Gate 17  43.158195 -77.607727   
2        5294                           TC- Gate 4  43.158357 -77.608449   
3        5306                          TC- Gate 15  43.158178 -77.607813   
4        5300                          TC- Gate 10  43.158396 -77.608160   
...       ...                                  ...        ...        ...   
40677    5083     Greece Ridge Mall Connection Hub  43.206124 -77.695661   
40678    2854                     Norfolk & Mimosa  43.096298 -77.750159   
40679    2856                Norfolk & Purple Leaf  43.097293 -77.750141   
40680    2853                  Norfolk & Forsythia  43.097953 -77.750103   
40681    2575  Monroe Community Coll Main Entrance  43.101842 -77.613798   

       RouteID                 RouteName Direction                   Time  \
0         

In [137]:
# Assume df is your DataFrame
# Convert 'ExpectedTime' to datetime
df['ExpectedTime'] = pd.to_datetime(df['ExpectedTime'], format='%m/%d/%Y %I:%M:%S %p')

# Generate random offsets between -5 and +5 minutes
random_offsets = [random.randint(-5, 10) for _ in range(len(df))]
df['Actual Arrival Time'] = df['ExpectedTime'] + pd.to_timedelta(random_offsets, unit='m')

# Display the DataFrame
print(df)

       StopID                             StopName        Lat        Lon  \
0        5296                           TC- Gate 6  43.158369 -77.608353   
1        5308                          TC- Gate 17  43.158195 -77.607727   
2        5294                           TC- Gate 4  43.158357 -77.608449   
3        5306                          TC- Gate 15  43.158178 -77.607813   
4        5300                          TC- Gate 10  43.158396 -77.608160   
...       ...                                  ...        ...        ...   
40677    5083     Greece Ridge Mall Connection Hub  43.206124 -77.695661   
40678    2854                     Norfolk & Mimosa  43.096298 -77.750159   
40679    2856                Norfolk & Purple Leaf  43.097293 -77.750141   
40680    2853                  Norfolk & Forsythia  43.097953 -77.750103   
40681    2575  Monroe Community Coll Main Entrance  43.101842 -77.613798   

       RouteID                 RouteName Direction        ExpectedTime  \
0           2

In [138]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Compute Delay (in seconds)
df['Delay'] = (pd.to_datetime(df['Actual Arrival Time']) - pd.to_datetime(df['ExpectedTime'])).dt.total_seconds() / 60


# Drop unnecessary columns
columns_to_drop = ['RunDate',  'Destination']
df = df.drop(columns=columns_to_drop)

# Encode categorical columns using Label Encoding
categorical_columns = ['StopName', 'RouteName', 'Direction','ExpectedTime', 'Actual Arrival Time',]
label_encoders = {}  # Dictionary to store label encoders for future reference

for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le  # Save the encoder for each column

# Save the label encoders
with open("label_encoders.pkl", "wb") as file:
    pickle.dump(label_encoders, file)

# Define features (X) and target (y)
X = df.drop(columns=['Delay'])
y = df['Delay']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display shapes of training and test sets
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")


Training features shape: (32545, 9)
Testing features shape: (8137, 9)
Training target shape: (32545,)
Testing target shape: (8137,)


In [139]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40682 entries, 0 to 40681
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   StopID               40682 non-null  int64  
 1   StopName             40682 non-null  int32  
 2   Lat                  40682 non-null  float64
 3   Lon                  40682 non-null  float64
 4   RouteID              40682 non-null  int64  
 5   RouteName            40682 non-null  int32  
 6   Direction            40682 non-null  int32  
 7   ExpectedTime         40682 non-null  int64  
 8   Actual Arrival Time  40682 non-null  int64  
 9   Delay                40682 non-null  float64
dtypes: float64(3), int32(3), int64(4)
memory usage: 2.6 MB


In [140]:
print(df)

       StopID  StopName        Lat        Lon  RouteID  RouteName  Direction  \
0        5296      1129  43.158369 -77.608353       24         16          2   
1        5308      1118  43.158195 -77.607727       17          8          2   
2        5294      1127  43.158357 -77.608449       21         13          2   
3        5306      1116  43.158178 -77.607813       15          6          2   
4        5300      1111  43.158396 -77.608160        6         23          2   
...       ...       ...        ...        ...      ...        ...        ...   
40677    5083       502  43.206124 -77.695661       20         12          2   
40678    2854       862  43.096298 -77.750159       18          9          2   
40679    2856       864  43.097293 -77.750141       18          9          2   
40680    2853       861  43.097953 -77.750103       18          9          2   
40681    2575       831  43.101842 -77.613798       17          8          2   

       ExpectedTime  Actual Arrival Tim

In [146]:
import pandas as pd
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import plot_importance
import matplotlib.pyplot as plt

# Assuming `X` and `y` are preprocessed features and target variable
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Initialize results storage
fold = 1
mae_list = []
mse_list = []
r2_list = []

for train_index, test_index in kf.split(X):
    print(f"\nFold {fold}")
    
    # Split the data
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Initialize the XGBoost Regressor
    xgb_model = XGBRegressor(
        n_estimators=1000,       # Number of trees
        learning_rate=0.1,       # Learning rate
        max_depth=8,             # Maximum depth of trees
        subsample=0.8,           # Fraction of samples used per tree
        colsample_bytree=0.8,    # Fraction of features used per tree
        gamma=0.8,               # Minimum split gain
        random_state=42          # Seed for reproducibility
    )
    
    # Train the model
    xgb_model.fit(X_train, y_train)
    
    # Make predictions on the test set
    y_pred = xgb_model.predict(X_test)
    
    # Evaluate the model
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Print results for the current fold
    print(f"Fold {fold} - Mean Absolute Error (MAE): {mae}")
    print(f"Fold {fold} - Mean Squared Error (MSE): {mse}")
    print(f"Fold {fold} - R-squared (R2): {r2}")
    
    # Append metrics to the lists
    mae_list.append(mae)
    mse_list.append(mse)
    r2_list.append(r2)
    
    fold += 1

# Calculate average metrics across all folds
print("\nOverall Performance:")
print(f"Average Mean Absolute Error (MAE): {sum(mae_list) / len(mae_list)}")
print(f"Average Mean Squared Error (MSE): {sum(mse_list) / len(mse_list)}")
print(f"Average R-squared (R2): {sum(r2_list) / len(r2_list)}")

# Optional: Feature importance visualization (based on the last model)
plot_importance(xgb_model)
plt.show()


Fold 1
Fold 1 - Mean Absolute Error (MAE): 2.204043780940651
Fold 1 - Mean Squared Error (MSE): 7.055803492380378
Fold 1 - R-squared (R2): 0.6581404333328571

Fold 2
Fold 2 - Mean Absolute Error (MAE): 2.2551849992888227
Fold 2 - Mean Squared Error (MSE): 7.27927441739613
Fold 2 - R-squared (R2): 0.6584866109184739

Fold 3
Fold 3 - Mean Absolute Error (MAE): 2.272938587745107
Fold 3 - Mean Squared Error (MSE): 7.3611569704639415
Fold 3 - R-squared (R2): 0.6570509939302456

Fold 4


KeyboardInterrupt: 

In [134]:
import pickle

final_model_filename = "xgb_final_model.pkl"
with open(final_model_filename, 'wb') as file:
    pickle.dump(xgb_model, file)

print(f"\nFinal model saved as {final_model_filename}")


Final model saved as xgb_final_model.pkl


In [135]:
# Calculate average delays
actual_avg_delay_seconds = y_test.mean() * 60  # Convert minutes to seconds
predicted_avg_delay_seconds = y_pred.mean() * 60  # Convert minutes to seconds

print(f"Average Actual Delay (seconds): {actual_avg_delay_seconds:.2f}")
print(f"Average Predicted Delay (seconds): {predicted_avg_delay_seconds:.2f}")


Average Actual Delay (seconds): -6.31
Average Predicted Delay (seconds): -4.16
